# D-Lys 第二嵌段候选筛选：可复现演示

**目标**：固定 D-Lys 功能块，从 17 种天然氨基酸中，对三个预注册机制域分别给出实验优先候选。

这不是“让大模型直接猜答案”：

1. 文献证据 + ICL 负责发现**应该比较哪些物化维度**；
2. AAindex / RDKit / peptides.py 提供**客观数值或结构属性**；
3. 有 ≥2 个独立指标的域用 **equal-weight TOPSIS**；只有 1 个独立轴的域不强行凑多指标算法；
4. Borda、criterion ablation、权重扰动和 Pareto 只做稳健性诊断。

> 本 Notebook 默认不重新调用 LLM API，避免录屏时随机性、网络延迟和密钥泄漏。它读取已经冻结的 ICL 结果，再现场重跑确定性的排名算法。

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# 自动定位项目根目录
def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'src' / 'dlys_multicriteria_ranking_v2.py').exists():
            return p
    raise FileNotFoundError('找不到项目根目录')

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from submission.src.demo_utils import resolve_demo_inputs, run_deterministic_ranking, load_ranking_tables, concise_final_table

print('Repository:', ROOT)
print('Python:', sys.version.split()[0])

## 1. 三个机制域先验冻结

不同类型侧链的组装机制并不相同，因此不把 17 个氨基酸粗暴塞进一个统一总分。域划分先于最终排名冻结。

In [ ]:
taxonomy = json.loads((ROOT / 'configs' / 'dlys_stage2_taxonomy_v3.json').read_text(encoding='utf-8'))
tax_rows = []
for d in taxonomy['domains']:
    tax_rows.append({
        'domain': d['domain_id'],
        '机制域': d['name_zh'],
        '候选': ' / '.join(d['candidates']),
        '比较逻辑': d['role_zh'],
    })
display(pd.DataFrame(tax_rows))

## 2. 读取冻结后的 ICL 指标发现结果

这里展示的是上游 RAG/ICL 已经得到并保存的结果。**ICL 不负责直接给 F/L/Q 打分，也不生成最终数值。**

录屏时这一部分展示“AI 推荐了哪些可比较维度”，而真正排名在后面现场重跑。

In [ ]:
discovery_run, intrinsic_csv = resolve_demo_inputs(ROOT)
print('Indicator discovery:', discovery_run)
print('Intrinsic table   :', intrinsic_csv)

ind = pd.read_csv(discovery_run / 'discovered_indicators.csv')
show_cols = [c for c in [
    'domain_id','domain_name_zh','indicator_name','measurement_target',
    'role','preferred_direction','quantifiable_single_residue','why_relevant_to_dlys_system'
] if c in ind.columns]
display(ind[show_cols])

## 3. 数据不是 LLM 编出来的

最终数值来自成熟工具：

- **AAindex**：已发表的单残基理化标度（如 Kyte–Doolittle hydropathy、side-chain volume）；
- **RDKit**：结构属性（芳香性、β-支化、酰胺等）；
- **peptides.py**：理论电荷及多种成熟氨基酸/肽描述符。

下面先看 17 种候选的 intrinsic property 表。

In [ ]:
intrinsic = pd.read_csv(intrinsic_csv)
keep = [c for c in [
    'aa','name_zh','kyte_doolittle','nominal_sidechain_charge_pH7_4',
    'beta_branched','has_sidechain_amide','sidechain_aromatic_ring_count',
    'sidechain_HBD_reference','sidechain_HBA_reference'
] if c in intrinsic.columns]
display(intrinsic[keep])

## 4. 现场重跑确定性排名

这一 Cell **不会重新调用 LLM**。它使用刚才冻结的指标发现结果与 intrinsic data，重新执行：

- 指标 → AAindex/RDKit/peptides.py 的确定性映射；
- 方向转换与冗余审计；
- D1 单轴决策（若只剩 1 个独立轴）；
- D2/D3 equal-weight TOPSIS；
- Borda、criterion ablation、±20% 权重扰动与 Pareto 诊断。

In [ ]:
ranking_run = run_deterministic_ranking(ROOT, discovery_run, intrinsic_csv)
print('Ranking output:', ranking_run)
tables = load_ranking_tables(ranking_run)
display(concise_final_table(tables))

## 5. ICL 推荐了什么，最后真正用了什么？

`criteria_used.csv` 是最重要的审计表：它把 ICL 提出的语言描述映射到成熟数据源，并记录方向、权重和用途。

如果某个指标无法安全映射、方向是 context-dependent、在域内为常数或与另一个轴完全序数冗余，它不会进入最终决策。

In [ ]:
criteria = tables['criteria_used'].copy()
cols = [c for c in [
    'domain_id','indicator_name','registry_key','family','tool','tool_id',
    'preferred_direction','transform','equal_weight','why_relevant_to_dlys_system'
] if c in criteria.columns]
display(criteria[cols])

if not tables['criteria_unresolved'].empty:
    display(Markdown('### 被审计掉、未进入最终决策的指标'))
    bad_cols = [c for c in [
        'domain_id','indicator_name','registry_key','tool','tool_id','exclusion_reason'
    ] if c in tables['criteria_unresolved'].columns]
    display(tables['criteria_unresolved'][bad_cols])

## 6. D1：芳香域为什么不是硬凑 TOPSIS？

F / W / Y 已经通过“芳香域”这一机制定义聚到一起。资格化与冗余审计后，只有 **Kyte–Doolittle hydropathy** 保持为一个独立、方向明确的正式排序轴。

因此：

- **芳香性**是域定义/门禁，不是第二张投票；
- 水化/极性等候选量若只重复 hydropathy 的域内序关系，则不重复计票；
- 不为了形式统一强行制造第二指标。

In [ ]:
d1 = tables['single_criterion_rankings']
if not d1.empty:
    display(d1[d1['domain_id']=='D1'].sort_values('primary_rank'))
else:
    print('当前 run 未生成 D1 单轴结果，请检查 discovery/ranking 版本。')

### D1 多疏水标度敏感性（诊断，不改主规则）

`peptides.py` 暴露多种已发表 hydrophobicity scales。下面统一每个标度的正负方向后，检查 F/W/Y 的第一名是否依赖于单一 hydropathy scale。

这个结果只用于说明**尺度敏感性**，不会用“多数投票”反向修改预注册的 Kyte–Doolittle 主规则。

In [ ]:
from src.dlys_hydropathy_scale_audit_v1 import load_peptides_hydrophobicity_tables, audit_hydropathy_tables

hydro_tables = load_peptides_hydrophobicity_tables()
hydro_detail, hydro_summary = audit_hydropathy_tables(hydro_tables)
display(hydro_summary)
display(hydro_detail[['scale','spearman_vs_kyte_doolittle','orientation_rule','raw_F','raw_W','raw_Y','winner']])

## 7. TOPSIS 到底怎么算？

对 D2 / D3，设决策矩阵为 $X=(x_{ij})$：每一行是候选，每一列是一个独立指标。

1. 先把“higher / lower / closer-to-zero / absence”等方向转成 benefit 或 cost；
2. 指标采用**等权**，不是让 LLM 赋权；
3. TOPSIS 构造正理想解 $A^+$ 和负理想解 $A^-$；
4. 计算每个候选到二者的距离 $D_i^+$ 与 $D_i^-$；
5. closeness：

$$C_i = \frac{D_i^-}{D_i^+ + D_i^-}$$

$C_i$ 越大，表示越接近综合理想状态。

所以 TOPSIS 与“把三个原始数直接相加”不是一回事；它保留多维距离信息。

## 8. D2：非芳香疏水-构象域

当前正式三轴：

1. hydropathy：越高越好；
2. β-branching：absence preferred；
3. side-chain volume：越高越好。

这里亮氨酸的优势应理解为三个机制轴共同形成的 **compromise optimum**。

In [ ]:
matrix = tables['decision_matrix_long']
d2_matrix = matrix[matrix['domain_id']=='D2'].pivot_table(
    index=['candidate','candidate_name_zh'], columns='registry_key', values='raw_value'
).reset_index()
display(d2_matrix)

d2_t = tables['topsis_rankings']
d2_b = tables['borda_rankings']
display(d2_t[d2_t['domain_id']=='D2'].sort_values('topsis_rank'))
display(d2_b[d2_b['domain_id']=='D2'].sort_values('borda_rank'))

In [ ]:
d2_plot = d2_t[d2_t['domain_id']=='D2'].sort_values('topsis_preference', ascending=False)
plt.figure(figsize=(7,4))
plt.bar(d2_plot['candidate'], d2_plot['topsis_preference'])
plt.ylabel('TOPSIS preference')
plt.xlabel('Candidate')
plt.title('D2 TOPSIS ranking')
plt.show()

### D2 稳健性如何理解？

- **Borda**：只看各指标名次，是独立的序数聚合诊断；
- **criterion ablation**：每次整个删掉一个机制轴，看 winner 是否变化；这是机制依赖性诊断，不是“稳定性分数”；
- **±20% weight perturbation**：三个机制轴都保留，仅在等权附近扰动，看结果是否依赖某个精确权重。

In [ ]:
diag = tables['decision_diagnostics']
display(diag[diag['domain_id']=='D2'])
display(tables['leave_one_criterion_out'][tables['leave_one_criterion_out']['domain_id']=='D2'])
display(tables['weight_perturbation_sensitivity'][tables['weight_perturbation_sensitivity']['domain_id']=='D2'])

## 9. D3：极性-氢键-构象相容域

当前正式三轴：

1. side-chain charge at pH 7.4：越接近 0 越好；
2. side-chain amide：presence preferred；
3. side-chain volume：越大越优，作为当前 interaction reach / side-chain extent proxy。

**重要边界**：Q 与 N 的电荷和酰胺化学很接近，Q>N 的主要算法差异来自第三个 size/reach proxy。因此它是“实验优先级”，不是证明 Q 在所有材料体系中普适优于 N。

In [ ]:
d3_matrix = matrix[matrix['domain_id']=='D3'].pivot_table(
    index=['candidate','candidate_name_zh'], columns='registry_key', values='raw_value'
).reset_index()
display(d3_matrix)

d3_t = d2_t[d2_t['domain_id']=='D3']
d3_b = d2_b[d2_b['domain_id']=='D3']
display(d3_t.sort_values('topsis_rank'))
display(d3_b.sort_values('borda_rank'))

# 单独查看 Q / N 的实际决策矩阵值
display(d3_matrix[d3_matrix['candidate'].isin(['Q','N'])])

In [ ]:
d3_plot = d3_t.sort_values('topsis_preference', ascending=False)
plt.figure(figsize=(8,4))
plt.bar(d3_plot['candidate'], d3_plot['topsis_preference'])
plt.ylabel('TOPSIS preference')
plt.xlabel('Candidate')
plt.title('D3 TOPSIS ranking')
plt.show()

display(diag[diag['domain_id']=='D3'])
display(tables['leave_one_criterion_out'][tables['leave_one_criterion_out']['domain_id']=='D3'])
display(tables['weight_perturbation_sensitivity'][tables['weight_perturbation_sensitivity']['domain_id']=='D3'])

## 10. 最终实验优先候选

本阶段的目标是**缩小实验空间**，不是宣称已经预测出完整长链配方的绝对最优解。

In [ ]:
final = concise_final_table(tables)
display(final)

selected = dict(zip(final['domain_id'], final['selected_candidate_name_zh']))
print('主实验候选：')
for did in ['D1','D2','D3']:
    print(f'  {did}: {selected.get(did)}')
print()
print('若实验预算允许：N（天冬酰胺）可作为 D3 的机制近邻对照/备选。')

## 汇报时一句话总结

> 我们没有让大模型直接猜哪个氨基酸最好，而是先用文献证据和上下文学习发现不同机制域需要比较的物化维度，再用 AAindex、RDKit 和 peptides.py 提供客观数据；多个独立指标时用等权 TOPSIS 进行候选优选，并用 Borda、机制轴消融和权重扰动做诊断。最终优先得到苯丙氨酸、亮氨酸和谷氨酰胺，作为后续长链与实验验证的三个代表候选。

**边界**：这是 candidate prioritization，不是监督机器学习对完整聚合物性能的直接预测。